# Project: Text Mining and Sentiment Analysis on IMDB Movie Reviews

## Introduction
This notebook presents a text-mining pipeline for binary sentiment classification on the IMDB Large Movie Review Dataset. It demonstrates data download & loading, tokenization and normalization, embeddings (Bag-of-Words and TF-IDF), model training (Linear SVM and Random Forest), and evaluation.

## Objectives
- Load and preprocess review texts from the IMDB dataset
- Compare tokenization and normalization approaches
- Evaluate Bag-of-Words vs TF-IDF embeddings and classifier performance

## Highlights
- Treebank tokenization and Porter stemming implemented
- Pipeline comparisons with accuracy metrics
- All code cells are runnable and documented for a portfolio presentation


Source:  
This project is based on an assignment of Project Management course at Stockholm University. 

In [ ]:
#install required libraries
!pip install pandas sklearn nltk

## Download and prepare data:

The code in this section downloads the [IMDB IMDB Large Movie Review Dataset]('https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz') which is the dataset you will be working on in this assignment.

In [ ]:
#import required libraries
import os
import tarfile
from urllib.request import urlretrieve

In [ ]:
#download and extract data
if not os.path.exists('aclImdb'):
    # download data:
    urlretrieve('https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz', 'aclImdb.tar.gz')

    # unzip data:
    with tarfile.open('aclImdb.tar.gz') as file:
        file.extractall('./')

## Some helper Functions:

Some helper functions are created first to use in further process.

Import the required libraries: pandas as pd for DataFrame handling; Literal, Tuple, Iterable for type annotations.



In [ ]:
import pandas as pd
from typing import Literal, Tuple, Iterable

Function for loading data into a pandas dataframe:

In [5]:
def load_data(split:Literal['train', 'test'], texts_per_class:int=500) -> pd.DataFrame:
    ''' Loads the data into a pandas dataframe.'''
    paths  = []
    labels = []

    for label in ('pos', 'neg'):
        # get all files in the folder:
        files = os.listdir(os.path.join('aclImdb', split, label))[:texts_per_class]

        # append them to the lists:
        paths.extend([os.path.join('aclImdb', split, label, f) for f in files])
        labels.extend([label] * len(files))

    return pd.DataFrame({'path':paths, 'label':labels})

Function for loading a specific text:

In [6]:
def load_text(path:str) -> str:
    ''' Reads a single text given the path. '''
    # read file from disk:
    with open(path, 'r', encoding='utf8') as file:
        s = file.read()

    return s

Function for iterating through multiple texts:

In [7]:
def iterate_texts(data:pd.DataFrame) -> Iterable[Tuple[str, str]]:
    ''' Iterates through a pandas dataframe. '''

    for path in data['path'].values:
        # read file from disk:
        with open(path, 'r', encoding='utf8') as file:
            text = file.read()

        yield text

Load the training and test data into DataFrames.

In [8]:
data_train = load_data('train')
data_test  = load_data('test')
data_train

,path,label
0,aclImdb\train\pos\0_9.txt,pos
1,aclImdb\train\pos\10000_8.txt,pos
2,aclImdb\train\pos\10001_10.txt,pos
3,aclImdb\train\pos\10002_7.txt,pos
4,aclImdb\train\pos\10003_8.txt,pos
...,...,...
995,aclImdb\train\neg\10446_2.txt,neg
996,aclImdb\train\neg\10447_1.txt,neg
997,aclImdb\train\neg\10448_1.txt,neg
998,aclImdb\train\neg\10449_4.txt,neg


### Accessing the texts:

In [ ]:
# load a single text
load_text(data_train.loc[0, 'path'])

'Bromwell High is a cartoon comedy. It ran at the same time as some other programs about school life, such as "Teachers". My 35 years in the teaching profession lead me to believe that Bromwell High\'s satire is much closer to reality than is "Teachers". The scramble to survive financially, the insightful students who can see right through their pathetic teachers\' pomp, the pettiness of the whole situation, all remind me of the schools I knew and their students. When I saw the episode in which a student repeatedly tried to burn down the school, I immediately recalled ......... at .......... High. A classic line: INSPECTOR: I\'m here to sack one of your teachers. STUDENT: Welcome to Bromwell High. I expect that many adults of my age think that Bromwell High is far fetched. What a pity that it isn\'t!'

In [10]:
# Sample code: iterate through all texts
for text in iterate_texts(data_train[:20]):
    print(text)

Bromwell High is a cartoon comedy. It ran at the same time as some other programs about school life, such as "Teachers". My 35 years in the teaching profession lead me to believe that Bromwell High's satire is much closer to reality than is "Teachers". The scramble to survive financially, the insightful students who can see right through their pathetic teachers' pomp, the pettiness of the whole situation, all remind me of the schools I knew and their students. When I saw the episode in which a student repeatedly tried to burn down the school, I immediately recalled ......... at .......... High. A classic line: INSPECTOR: I'm here to sack one of your teachers. STUDENT: Welcome to Bromwell High. I expect that many adults of my age think that Bromwell High is far fetched. What a pity that it isn't!
Homelessness (or Houselessness as George Carlin stated) has been an issue for years but never a plan to help those on the street that were once considered human who did everything from going to

Now, tokenize lowercase text and split on whitespace.

**White-Space tokenization:**
Split text on spaces very fast but ignores punctuation and contractions.

In [11]:
def tokenize(text:str):
    ''' An example tokenization function. '''

    # simple white-space tokenization:
    return text.lower().split()


**Bag-of-words Embedding:**

See documentation of [sklearn.feature_extraction.text.CountVectorizer](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html)

Bag-of-Words embedding:

We create a `CountVectorizer` that converts each document (loaded from file paths using `load_text`) into a sparse vector of token counts using the `tokenize` function defined above. This yields a fixed-length numeric representation where each feature corresponds to a token and values are term frequencies.

We fit the vectorizer on the training set to learn the vocabulary, then transform both training and test sets into sparse feature matrices suitable for classifier training and evaluation.

In [12]:
from sklearn.feature_extraction.text import CountVectorizer

# create a simple bag of words embedding:
bow = CountVectorizer(

    # the next line converts the filepaths to the actual texts:
    preprocessor = load_text,

    # tokenization function from above:
    tokenizer = tokenize

)

# train the embedding:
embeddings_train = bow.fit_transform(data_train['path'].values)

# vectorize test data:
embeddings_test = bow.transform(data_test['path'].values)

c:\Users\Lenovo\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\feature_extraction\text.py:521: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


**Classification with Linear SVM**

We train a linear Support Vector Machine (`LinearSVC`) on the document embeddings produced above (for example, the Bag-of-Words `embeddings_train`). Linear SVMs are a strong baseline for text classification because they handle high-dimensional, sparse feature spaces efficiently and often deliver competitive accuracy.

The code below fits the SVM using `embeddings_train` and `data_train['label']`, then predicts labels for `embeddings_test`. We evaluate performance using `accuracy_score`, which returns the proportion of correctly predicted test labels. Ensure your feature matrices are numeric (dense arrays or sparse matrices) and that the label arrays align with the training instances before fitting.

In [13]:
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score

svm = LinearSVC()

# train classifier:
svm.fit(embeddings_train, data_train['label'].values)

# test classifier:
predictions = svm.predict(embeddings_test)

# Calculate Accuracy:
print('Accuracy:', accuracy_score(data_test['label'].values, predictions))

Accuracy: 0.759


c:\Users\Lenovo\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\svm\_base.py:1235: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


# Text-Mining Workflow

This notebook now walks again through a complete, reproducible text-mining workflow:

- Tokenization & normalization — we compare simple white-space tokenization with the Treebank tokenizer and optional Porter stemming to show how token choices affect features.
- Embedding methods — we demonstrate Bag-of-Words (CountVectorizer) and TF-IDF (TfidfVectorizer), including vocabulary learning, stop-word removal, and n-gram configuration.
- Model training & pipelines — we train and compare models (Linear SVM and Random Forest) using consistent train/test splits and simple sklearn pipelines to ensure reproducibility.
- Evaluation & analysis — models are evaluated with accuracy (and can be extended to other metrics); limitations and practical considerations are discussed to guide improvements.

Together these sections provide an end-to-end example you can run, adapt, and include in a portfolio to demonstrate applied NLP and model evaluation. 

## 1.Preparation

Some codes below are executed to print some texts from the dataset, and to understand the above functions and the dataframe.

In [14]:
#to know the above 3 rows of training set
data_train[0:3]

,path,label
0,aclImdb\train\pos\0_9.txt,pos
1,aclImdb\train\pos\10000_8.txt,pos
2,aclImdb\train\pos\10001_10.txt,pos


In [15]:
# to know the path of the text file in 3rd row of training data. The path is returned as string.
data_test.loc[3, 'path']

'aclImdb\\test\\pos\\10002_8.txt'

In [16]:
#to load the text in the txt file at 3rd row of training data
#using the function load_text to know how it returns
load_text(data_test.loc[3, 'path'])

"I saw this film in a sneak preview, and it is delightful. The cinematography is unusually creative, the acting is good, and the story is fabulous. If this movie does not do well, it won't be because it doesn't deserve to. Before this film, I didn't realize how charming Shia Lebouf could be. He does a marvelous, self-contained, job as the lead. There's something incredibly sweet about him, and it makes the movie even better. The other actors do a good job as well, and the film contains moments of really high suspense, more than one might expect from a movie about golf. Sports movies are a dime a dozen, but this one stands out. <br /><br />This is one I'd recommend to anyone."

In [17]:
#run the function, iterate_texts, to know its return which is an object
# the object relates to the first 3 rows of training data
iterate_texts(data_train[0:2])

<generator object iterate_texts at 0x00000152F1EB5640>

In [18]:
#extract the texts in the text files of paths at first 5 rows of training data
# use for loop to extract the texts from the object returned by iterate_texts
for text in iterate_texts(data_train[:5]):
    print(text)

Bromwell High is a cartoon comedy. It ran at the same time as some other programs about school life, such as "Teachers". My 35 years in the teaching profession lead me to believe that Bromwell High's satire is much closer to reality than is "Teachers". The scramble to survive financially, the insightful students who can see right through their pathetic teachers' pomp, the pettiness of the whole situation, all remind me of the schools I knew and their students. When I saw the episode in which a student repeatedly tried to burn down the school, I immediately recalled ......... at .......... High. A classic line: INSPECTOR: I'm here to sack one of your teachers. STUDENT: Welcome to Bromwell High. I expect that many adults of my age think that Bromwell High is far fetched. What a pity that it isn't!
Homelessness (or Houselessness as George Carlin stated) has been an issue for years but never a plan to help those on the street that were once considered human who did everything from going to

In [19]:
#for clarity, insert a space line between texts
for text in iterate_texts(data_train[:5]):
    print(text)
    print(' ')

Bromwell High is a cartoon comedy. It ran at the same time as some other programs about school life, such as "Teachers". My 35 years in the teaching profession lead me to believe that Bromwell High's satire is much closer to reality than is "Teachers". The scramble to survive financially, the insightful students who can see right through their pathetic teachers' pomp, the pettiness of the whole situation, all remind me of the schools I knew and their students. When I saw the episode in which a student repeatedly tried to burn down the school, I immediately recalled ......... at .......... High. A classic line: INSPECTOR: I'm here to sack one of your teachers. STUDENT: Welcome to Bromwell High. I expect that many adults of my age think that Bromwell High is far fetched. What a pity that it isn't!
 
Homelessness (or Houselessness as George Carlin stated) has been an issue for years but never a plan to help those on the street that were once considered human who did everything from going 

In [20]:
#to know the result of bag-of-winds(BOW) embeddings, print the embeddings_train which is the return of BOW embeddings of training data 
#it returns compressed sparse row sparse matix
embeddings_train

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 150666 stored elements and shape (1000, 30586)>

In [21]:
#to know the result of tokenization of a text form path of 2nd row of training data,
#first, text is extracted using load_text function, then, this text is tokenized with tokenize function
#after printing the return 'text1' of tokenize function, it generate the list of tokens form the text
text1 = tokenize(load_text(data_train.loc[1, 'path']))
text1

['homelessness',
 '(or',
 'houselessness',
 'as',
 'george',
 'carlin',
 'stated)',
 'has',
 'been',
 'an',
 'issue',
 'for',
 'years',
 'but',
 'never',
 'a',
 'plan',
 'to',
 'help',
 'those',
 'on',
 'the',
 'street',
 'that',
 'were',
 'once',
 'considered',
 'human',
 'who',
 'did',
 'everything',
 'from',
 'going',
 'to',
 'school,',
 'work,',
 'or',
 'vote',
 'for',
 'the',
 'matter.',
 'most',
 'people',
 'think',
 'of',
 'the',
 'homeless',
 'as',
 'just',
 'a',
 'lost',
 'cause',
 'while',
 'worrying',
 'about',
 'things',
 'such',
 'as',
 'racism,',
 'the',
 'war',
 'on',
 'iraq,',
 'pressuring',
 'kids',
 'to',
 'succeed,',
 'technology,',
 'the',
 'elections,',
 'inflation,',
 'or',
 'worrying',
 'if',
 "they'll",
 'be',
 'next',
 'to',
 'end',
 'up',
 'on',
 'the',
 'streets.<br',
 '/><br',
 '/>but',
 'what',
 'if',
 'you',
 'were',
 'given',
 'a',
 'bet',
 'to',
 'live',
 'on',
 'the',
 'streets',
 'for',
 'a',
 'month',
 'without',
 'the',
 'luxuries',
 'you',
 'once',


In [22]:
# to know the type of return of tokenize function and the number of tokens
print('type of tokenized result:',type(text1), 'it includes',len(text1),'tokens.')

type of tokenized result: <class 'list'> it includes 428 tokens.


## 2. Tokenization

TreebankWord Tokenizer will be used for implementing tokenzation

In [24]:
# Import TreebankWordTokenizer and TreebankWordDetokenizer from nltk
from nltk.tokenize.treebank import TreebankWordTokenizer, TreebankWordDetokenizer
import nltk

# Download the punkt module
nltk.download('punkt')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Lenovo\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

Two texts will be tokenzied by using TreebankWord tokenizer as follows.

In [25]:
# Create an example text
s = "I'm excited to watch this film! It's well-reviewed."

# Use TreebankWordTokenizer to tokenize the text
t = TreebankWordTokenizer().tokenize(s)
print("Tokens:", t)

# Use TreebankWordDetokenizer to reassemble the text
detokenized_text = TreebankWordDetokenizer().detokenize(t)
print("Detokenized text:", detokenized_text)

Tokens: ['I', "'m", 'excited', 'to', 'watch', 'this', 'film', '!', 'It', "'s", 'well-reviewed', '.']
Detokenized text: I'm excited to watch this film! It's well-reviewed.


In [26]:
# Create an example text
s = "Best dramatic hobo lady I have ever seen, and love scenes in clothes warehouse are second to none."

# Use TreebankWordTokenizer to tokenize the text
t = TreebankWordTokenizer().tokenize(s)
print("Tokens:", t)

# Use TreebankWordDetokenizer to reassemble the text
detokenized_text = TreebankWordDetokenizer().detokenize(t)
print("Detokenized text:", detokenized_text)

Tokens: ['Best', 'dramatic', 'hobo', 'lady', 'I', 'have', 'ever', 'seen', ',', 'and', 'love', 'scenes', 'in', 'clothes', 'warehouse', 'are', 'second', 'to', 'none', '.']
Detokenized text: Best dramatic hobo lady I have ever seen, and love scenes in clothes warehouse are second to none.


In [27]:
# Create an example text
s = "Like one of the previous commenters said, this had the foundations of a great movie but something happened on the way to delivery."

# Use TreebankWordTokenizer to tokenize the text
t = TreebankWordTokenizer().tokenize(s)
print("Tokens:", t)

# Use TreebankWordDetokenizer to reassemble the text
detokenized_text = TreebankWordDetokenizer().detokenize(t)
print("Detokenized text:", detokenized_text)

Tokens: ['Like', 'one', 'of', 'the', 'previous', 'commenters', 'said', ',', 'this', 'had', 'the', 'foundations', 'of', 'a', 'great', 'movie', 'but', 'something', 'happened', 'on', 'the', 'way', 'to', 'delivery', '.']
Detokenized text: Like one of the previous commenters said, this had the foundations of a great movie but something happened on the way to delivery.


Then, for text normalization, stemming by porter stemmer will be used. Then, two texts will be tokenized and normalized as follows. Improvement in tokenzation using normalization can be seen and it will be explained in the report.

In [28]:
# Import the Porterstemmer class from NLTK
from nltk.stem.porter import PorterStemmer


In [29]:
# Initialize the tokenizer and stemmer
tokenizer = TreebankWordTokenizer()
stemmer = PorterStemmer()

# Sample texts
text1 = "Like one of the previous commenters said, this had the foundations of a great movie but something happened on the way to delivery."
text2 = "Best dramatic hobo lady I have ever seen, and love scenes in clothes warehouse are second to none."

# Tokenize the texts
tokens_text1 = tokenizer.tokenize(text1)
tokens_text2 = tokenizer.tokenize(text2)

# Print tokenized texts
print("Tokens from Text 1:", tokens_text1)
print("Tokens from Text 2:", tokens_text2)

# Apply stemming to each token
stemmed_text1 = [stemmer.stem(token) for token in tokens_text1]
stemmed_text2 = [stemmer.stem(token) for token in tokens_text2]

# Print the stemmed results
print("Stemmed Text 1:", stemmed_text1)
print("Stemmed Text 2:", stemmed_text2)

Tokens from Text 1: ['Like', 'one', 'of', 'the', 'previous', 'commenters', 'said', ',', 'this', 'had', 'the', 'foundations', 'of', 'a', 'great', 'movie', 'but', 'something', 'happened', 'on', 'the', 'way', 'to', 'delivery', '.']
Tokens from Text 2: ['Best', 'dramatic', 'hobo', 'lady', 'I', 'have', 'ever', 'seen', ',', 'and', 'love', 'scenes', 'in', 'clothes', 'warehouse', 'are', 'second', 'to', 'none', '.']
Stemmed Text 1: ['like', 'one', 'of', 'the', 'previou', 'comment', 'said', ',', 'thi', 'had', 'the', 'foundat', 'of', 'a', 'great', 'movi', 'but', 'someth', 'happen', 'on', 'the', 'way', 'to', 'deliveri', '.']
Stemmed Text 2: ['best', 'dramat', 'hobo', 'ladi', 'i', 'have', 'ever', 'seen', ',', 'and', 'love', 'scene', 'in', 'cloth', 'warehous', 'are', 'second', 'to', 'none', '.']


Accuracies on test set

To compare the accuracy of the test set, the dataset is tokenized in 3 ways: using white space tokenization, custom tokenization(TreebankWord tokenizer) and custom tokenzation with normalization. Then, random forest classifer model is trained and tested in dataset resulted from each tokenziaton. Then, the accuracies of the 3 models are calculated and compared with explanation in a table. The table can be seen in the report.

In [30]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Load train and test data
data_train = load_data('train')
data_test  = load_data('test')

# Custom white-space tokenizer
def tokenize(text: str):
    '''An example tokenization function with white-space tokenization.'''
    return text.lower().split()

# Custom tokenizer without normalization
def custom_tokenizer(text):
    tokenizer = TreebankWordTokenizer()
    return tokenizer.tokenize(text)

# Custom tokenizer with stemming
def custom_tokenizer_with_stemming(text):
    tokenizer = TreebankWordTokenizer()
    stemmer = PorterStemmer()
    tokens = tokenizer.tokenize(text)
    return [stemmer.stem(token) for token in tokens]

# Define pipelines
def create_pipeline(tokenizer):
    pipeline = Pipeline([
        ('vectorizer', CountVectorizer(tokenizer=tokenizer, token_pattern=None)),
        ('classifier', RandomForestClassifier())
    ])
    return pipeline

# Train the model
def train_model(pipeline, data_train, data_test):
    X_train = list(iterate_texts(data_train)) # This line is changed
    y_train = data_train['label']
    X_test = list(iterate_texts(data_test)) # This line is changed
    y_test = data_test['label']

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    return accuracy


# Create pipelines for different tokenizations
pipeline_default = create_pipeline(tokenize)  # Default white space tokenization
pipeline_custom = create_pipeline(custom_tokenizer)  # Improved tokenizer
pipeline_custom_stemming = create_pipeline(custom_tokenizer_with_stemming)  # Improved tokenizer + stemming

# Compute accuracy for each pipeline
accuracy_default = train_model(pipeline_default, data_train, data_test)
accuracy_custom = train_model(pipeline_custom, data_train, data_test)
accuracy_custom_stemming = train_model(pipeline_custom_stemming, data_train, data_test)


# Table summary
comparison_data = {
    'Tokenization Approach': ['Default White Space', 'Improved Tokenization', 'Improved Tokenization + Stemming'],
    'Accuracy on Test Set': [accuracy_default, accuracy_custom, accuracy_custom_stemming]
}
comparison_data

{'Tokenization Approach': ['Default White Space',
  'Improved Tokenization',
  'Improved Tokenization + Stemming'],
 'Accuracy on Test Set': [0.772, 0.793, 0.785]}

#### Comparison of Accuracy  
White space tokenization - 0.778  
TreebankWordTokenizer - 0.789  
TreebankWordTokenizer with Normalization - 0.793  

While the accuracy is not significantly changing, the results show that improving tokenization
quality and adding normalization steps make the text data more meaningful for the machine
learning model, leading to better predictions. Normalization improves token consistency,
reducing word variation. For example, "run," "running," and "ran" would all be treated as the
same token, enhancing the model’s ability to generalize patterns in the data. White space
tokenization can miss separating punctuation from words, for eg., "don't" becomes "don't"
instead of "do" and "n't" which may reduce the model’s accuracy.

## 3. Embedding

CountVectorizer of sklearn is learnt by reading its documentation and testing in the jupyter notebook.

TF-IDF embeddings is implemented using TfidfVectorizer of sklearn as follows.

In [31]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Create a new instance of TfidfVectorizer with basic configuration
tfidf_vectorizer = TfidfVectorizer(
    tokenizer=tokenize, # Use white-space tokenizer above
    preprocessor = load_text, 
    stop_words='english',  # Remove English stop words
    ngram_range=(1, 2)  # Unigrams and bigrams
)

Then, transform the text for training and testing into vectorized form using the TD-IDF embeddings, making them ready for developing prediction model.

In [32]:
X_train_tfidf = tfidf_vectorizer.fit_transform(data_train['path'].values)
X_test_tfidf = tfidf_vectorizer.transform(data_test['path'].values)

c:\Users\Lenovo\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\feature_extraction\text.py:521: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [33]:
#print the training data to know its type and shape
X_train_tfidf

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 221289 stored elements and shape (1000, 136397)>

In [34]:
#print the testing data to know its type and shape
X_test_tfidf

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 90523 stored elements and shape (1000, 136397)>

Then, a classifier model, linear SVM is developed by training the training data, and its performance, accuracy is tested on testing data.

In [35]:
from sklearn.svm import LinearSVC

# Train the classifier on the TF-IDF vectors
svm_model_tfidf = LinearSVC()
svm_model_tfidf.fit(X_train_tfidf, data_train['label'].values)

# Evaluate on the test set
y_pred_tfidf = svm_model_tfidf.predict(X_test_tfidf)

# Calculate accuracy
from sklearn.metrics import accuracy_score
accuracy_tfidf = accuracy_score(data_test['label'].values, y_pred_tfidf)

print(f"TF-IDF Model Accuracy: {accuracy_tfidf:.4f}")

TF-IDF Model Accuracy: 0.8190


The accuracy of linear SVM model trained on data from TF-IDF embeddings is 0.819

The accuracy of the developed classifier from Bag-of-Words embedding and TF-IDF are compared.  
Bag-of-Words Embedding -  0.759  
TF-IDF Embedding -  0.819

The Bag-of-Words embedding achieved an
accuracy of 75.9%. This shows that the
frequency of words was properly utilized in
the training data to predict the labels of the
test data. Since this is pretty straightforward,
it catches important words leading to good
accuracy, but it may not capture enough
semantic information, producing lower
accuray than TF-IDF embedding.

TF_IDF embedding achieved an accuracy of
81%, which is higher than that of
Bag-of-Words Embedding. It is because it
provides a good representation of words
through weighing them according to their
frequency in the dataset. By this way, it
emphasizes on meaningful terms and captures
more relevant features, producing better
accuracy.  





## Conclusion
This notebook demonstrates a complete text-mining workflow for the IMDB movie review dataset, from data loading and tokenizer design to embedding creation and model evaluation.
The results show that TF-IDF embeddings combined with a linear SVM produce stronger performance than a simple Bag-of-Words representation, while improved tokenization and normalization also help extract cleaner features.
Future improvements could include more advanced preprocessing, additional classifiers, and evaluation with precision, recall, or F1 score to better understand model behavior.